# Mission 1 신고자 성별 분류 — 최종 모델 학습

**최종 모델**: ECAPA-TDNN(SpeechBrain `spkrec-ecapa-voxceleb`, 가중치 고정) 임베딩 + 로지스틱 회귀

| 단계 | 내용 |
|---|---|
| 입력 | 신고자(speaker=1) 발화 구간 (`startAt`, `endAt`, `speaker`만 사용) |
| 전처리 | 8kHz → 16kHz 리샘플링, 0.3초 미만 발화 제외 |
| 특징 추출 | 발화별 ECAPA 임베딩(192차원), L2 정규화 |
| 통화 벡터 | 발화 길이 가중 평균 |
| 분류기 | StandardScaler + LogisticRegression(C=1), threshold 0.5 |
| 결과 | Validation **Accuracy 0.9821 / F1 0.9834** (통화 단위, 3,640통화) |

- 학습: Training(서울) 29,200통화만 사용, Validation은 집계 방식·하이퍼파라미터 선정과 성능 확인에만 사용 (역전파·학습 미사용)
- 환경: Google Colab, Tesla T4, seed 42

## 0. 환경 및 데이터 준비

In [ ]:
!nvidia-smi

Thu Sep 24 06:37:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Drive에 올린 원본 zip을 Colab 로컬로 복사 후 압축 해제 (Drive 개별 파일 접근보다 빠름)
import glob, os
zips = sorted(glob.glob("/content/drive/MyDrive/DCC_원본_zip/*.zip"))
os.makedirs("/content/data", exist_ok=True)
print(len(zips), "개")

for z in zips:
    print("처리 중:", os.path.basename(z))
    !cp "{z}" /content/tmp.zip
    !unzip -q -o /content/tmp.zip -d /content/data/
    !rm /content/tmp.zip

11 개
처리 중: 대학부 데이터-20260904T131135Z-1-001.zip
처리 중: 대학부 데이터-20260904T131135Z-1-002.zip
처리 중: 대학부 데이터-20260904T131135Z-1-003.zip
처리 중: 대학부 데이터-20260904T131135Z-1-004.zip
처리 중: 대학부 데이터-20260904T131135Z-1-005.zip
처리 중: 대학부 데이터-20260904T131135Z-1-006.zip
처리 중: 대학부 데이터-20260904T131135Z-1-007.zip
처리 중: 대학부 데이터-20260904T131135Z-1-008.zip
처리 중: 대학부 데이터-20260904T131135Z-1-009.zip
처리 중: 대학부 데이터-20260904T131135Z-1-010.zip
처리 중: 대학부 데이터-20260904T131135Z-1-011.zip


In [ ]:
# 데이터 경로 (서울 구급 데이터)
DATA_ROOT = "/content/data/대학부 데이터"
TRAIN_WAV  = f"{DATA_ROOT}/Training/1.원천데이터/TS_서울_구급"
TRAIN_JSON = f"{DATA_ROOT}/Training/2.라벨링데이터/TL_서울_구급"
VAL_WAV    = f"{DATA_ROOT}/Validation/1.원천데이터/VS_서울_구급"
VAL_JSON   = f"{DATA_ROOT}/Validation/2.라벨링데이터/VL_서울_구급"

In [ ]:
# wav와 json이 파일명(확장자 제외)으로 1:1 매칭되는지 확인
import glob
import os
stem = lambda p: os.path.splitext(os.path.basename(p))[0]
for name, wav_dir, json_dir in [("train", TRAIN_WAV, TRAIN_JSON), ("val", VAL_WAV, VAL_JSON)]:
    w = {stem(p) for p in glob.glob(f"{wav_dir}/**/*.wav", recursive=True)}
    j = {stem(p) for p in glob.glob(f"{json_dir}/**/*.json", recursive=True)}
    print(name, "매칭:", len(w & j), "/ wav만:", len(w - j), "/ json만:", len(j - w))

train 매칭: 29200 / wav만: 0 / json만: 0
val 매칭: 3640 / wav만: 0 / json만: 0


## 1. 공통 설정 (seed, 저장 경로, 평가 함수)

In [ ]:
import os, random, numpy as np, torch, pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# 경로와 seed
SEEDS = [42, 123, 2026]
BASE_DIR = "/content/drive/MyDrive/DCC/mission1"
DIRS = {k: os.path.join(BASE_DIR, k) for k in ["ckpt", "preds", "history"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

LABEL_MAP = {"F": 1, "M": 0}   # 공통 규칙: F=1, M=0

def set_seed(seed):
    """재현성을 위한 seed 고정"""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_prefix(model_name, setting, seed):
    """결과 파일 이름 규칙: m1_{모델}_{설정}_s{seed}"""
    return f"m1_{model_name}_{setting}_s{seed}"

def evaluate_calls(call_ids, probs, labels, threshold=0.5):
    """발화별 여성 확률을 통화 단위로 평균한 뒤 threshold 0.5로 판정해 Accuracy, F1 계산"""
    utt = pd.DataFrame({"call_id": call_ids, "prob_F": probs, "true": labels})
    call = (utt.groupby("call_id")
               .agg(prob_F=("prob_F", "mean"), true=("true", "first"))
               .reset_index())
    call["pred"] = (call["prob_F"] >= threshold).astype(int)
    acc = accuracy_score(call["true"], call["pred"])
    f1 = f1_score(call["true"], call["pred"])   # F(=1) 기준
    return acc, f1, utt, call

## 2. 사전학습 ECAPA-TDNN 불러오기

VoxCeleb으로 사전학습된 화자 임베딩 모델. 가중치는 고정하고 특징 추출기로만 사용합니다.

In [ ]:
!pip install -q speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 41.5 MB/s eta 0:00:00


In [ ]:
# GPU 사용 가능 시 GPU, 아니면 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

# 최초 1회 Hugging Face에서 받아 Drive(ECAPA_DIR)에 저장 → 제출 시 이 폴더를 로컬 가중치로 사용
ECAPA_DIR = os.path.join(BASE_DIR, "pretrained", "ecapa")
ecapa = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir=ECAPA_DIR,
    run_opts={"device": str(device)},
)
ecapa.eval()
print(os.listdir(ECAPA_DIR))

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 83.3MB            

embedding_model.ckpt: downloading bytes:           |  0.00B            

...

classifier.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


label_encoder.txt:   0%|          | 0.00/129k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


['hyperparams.yaml', 'embedding_model.ckpt', 'mean_var_norm_emb.ckpt', 'classifier.ckpt', 'label_encoder.ckpt']


## 3. 신고자 발화 임베딩 추출

통화별로 신고자 발화를 잘라 ECAPA 임베딩을 뽑고, 2,000통화 단위로 `.npz`에 저장합니다. (Colab 끊김 대비 이어서 실행 가능)

In [ ]:
import json
import soundfile as sf
import torchaudio.functional as AF
from tqdm.auto import tqdm

SR = 8000       # 원본 오디오 샘플링레이트
MIN_SEC = 0.3   # 0.3초 미만 발화는 임베딩이 불안정해 제외
EMB_DIR = os.path.join(BASE_DIR, "emb", "ecapa")
os.makedirs(EMB_DIR, exist_ok=True)

def load_call(json_path, wav_dir):
    """통화 하나: 16kHz로 리샘플링한 신고자 발화 목록과 라벨 반환"""
    with open(json_path, encoding="utf-8") as f:
        lab = json.load(f)
    cid = stem(json_path)
    audio, _ = sf.read(os.path.join(wav_dir, cid + ".wav"), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio16 = AF.resample(torch.tensor(audio).to(device), SR, 16000)   # 통화 전체를 한 번에 리샘플링

    segs = []
    for u in lab["utterances"]:
        if u["speaker"] != 1:
            continue
        s = audio16[int(u["startAt"] * 16): int(u["endAt"] * 16)]      # 16kHz: 1ms = 16샘플
        if len(s) > 0:
            segs.append(s)
    kept = [s for s in segs if len(s) >= MIN_SEC * 16000]
    if not kept and segs:
        kept = [max(segs, key=len)]    # 전부 짧으면 가장 긴 것 하나는 유지
    return cid, kept, LABEL_MAP[lab["gender"]]

@torch.no_grad()
def embed_segments(segs):
    """발화 여러 개를 padding해서 한 번에 임베딩"""
    lens = torch.tensor([len(s) for s in segs], dtype=torch.float32)
    batch = torch.nn.utils.rnn.pad_sequence(segs, batch_first=True)
    emb = ecapa.encode_batch(batch, (lens / lens.max()).to(device))
    return emb.squeeze(1).cpu().numpy()

def extract_split(split, json_dir, wav_dir, chunk_size=2000):
    """split 전체를 chunk_size 통화씩 나눠 임베딩·통화ID·라벨·발화 길이를 npz로 저장"""
    paths = sorted(glob.glob(f"{json_dir}/**/*.json", recursive=True))
    for ci in range(0, len(paths), chunk_size):
        out = os.path.join(EMB_DIR, f"{split}_{ci // chunk_size:03d}.npz")
        if os.path.exists(out):
            continue                                  # 이미 저장된 묶음은 건너뜀
        embs, cids, ys, durs = [], [], [], []
        for p in tqdm(paths[ci:ci + chunk_size], desc=f"{split} {ci // chunk_size:03d}"):
            cid, segs, y = load_call(p, wav_dir)
            if not segs:
                continue
            e = embed_segments(segs)
            embs.append(e)
            cids += [cid] * len(e); ys += [y] * len(e)
            durs += [len(s) / 16000 for s in segs]
        np.savez(out, emb=np.concatenate(embs), call_id=np.array(cids),
                 y=np.array(ys), dur=np.array(durs))
        print("저장:", out, "발화 수:", len(cids))

In [ ]:
extract_split("val", VAL_JSON, VAL_WAV)
extract_split("train", TRAIN_JSON, TRAIN_WAV)

val 000:   0%|          | 0/2000 [00:00<?, ?it/s]

저장: /content/drive/MyDrive/DCC/mission1/emb/ecapa/val_000.npz 발화 수: 31346


val 001:   0%|          | 0/1640 [00:00<?, ?it/s]

저장: /content/drive/MyDrive/DCC/mission1/emb/ecapa/val_001.npz 발화 수: 26174


train 000:   0%|          | 0/2000 [00:00<?, ?it/s]

...

저장: /content/drive/MyDrive/DCC/mission1/emb/ecapa/train_012.npz 발화 수: 33684


train 013:   0%|          | 0/2000 [00:00<?, ?it/s]

저장: /content/drive/MyDrive/DCC/mission1/emb/ecapa/train_013.npz 발화 수: 33355


train 014:   0%|          | 0/1200 [00:00<?, ?it/s]

저장: /content/drive/MyDrive/DCC/mission1/emb/ecapa/train_014.npz 발화 수: 17654


## 4. 임베딩 불러오기

In [ ]:
def load_emb(split):
    """저장된 npz 묶음을 하나로 합침"""
    files = sorted(glob.glob(os.path.join(EMB_DIR, f"{split}_*.npz")))
    parts = [np.load(f) for f in files]
    return {k: np.concatenate([p[k] for p in parts]) for k in ["emb", "call_id", "y", "dur"]}

tr, va = load_emb("train"), load_emb("val")
for name, s in [("train", tr), ("val", va)]:
    print(name, s["emb"].shape, "콜 수:", len(np.unique(s["call_id"])), "F 비율:", round(s["y"].mean(), 3))

train (457689, 192) 콜 수: 29200 F 비율: 0.539
val (57520, 192) 콜 수: 3640 F 비율: 0.548


In [ ]:
def save_preds(prefix, utt_df, call_df):
    """발화 단위·통화 단위 예측 결과를 csv로 저장"""
    utt_df.to_csv(os.path.join(DIRS["preds"], f"{prefix}_utt.csv"), index=False)
    call_df.to_csv(os.path.join(DIRS["preds"], f"{prefix}_call.csv"), index=False)

## 5. 통화 단위 집계 방식 · 하이퍼파라미터 선정

Validation으로 비교해 최종 설정을 고릅니다. (대회 Q&A 기준 허용 범위)

| 비교 | 결과 | 선택 |
|---|---|---|
| 발화 단위 분류 vs 통화 평균 | 0.9758 vs 0.9808 | 통화 단위 |
| 로지스틱 회귀 C (0.01~10) | 0.9808~0.9813, 차이 미미 | C=1 (기본값) |
| 평균 / 길이 가중 평균 / 평균+표준편차 | 0.9808 / **0.9821** / 0.9794 | 길이 가중 평균 |

In [ ]:
# (a) 발화마다 분류한 뒤 통화 단위로 확률 평균
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.pipeline import make_pipeline

Xtr, Xva = normalize(tr["emb"]), normalize(va["emb"])   # 임베딩 길이를 1로 맞춤
clf_a = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf_a.fit(Xtr, tr["y"])

p_a = clf_a.predict_proba(Xva)[:, 1]
acc_a, f1_a, utt_a, call_a = evaluate_calls(va["call_id"], p_a, va["y"])
save_preds(make_prefix("ecapa", "lrutt", 42), utt_a, call_a)
print(f"(a) 발화 단위  acc={acc_a:.4f}  f1={f1_a:.4f}")

(a) 발화 단위  acc=0.9758  f1=0.9778


In [ ]:
# (b) 통화 안의 발화 임베딩을 평균해 통화당 벡터 1개로 분류
def call_mean(s):
    df = pd.DataFrame(normalize(s["emb"]))
    df["call_id"], df["y"] = s["call_id"], s["y"]
    g = df.groupby("call_id")
    return g.mean(numeric_only=True).drop(columns="y").values, g["y"].first().values, g.size().index.values

Xtr_c, ytr_c, _ = call_mean(tr)
Xva_c, yva_c, ids_va = call_mean(va)

clf_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf_b.fit(Xtr_c, ytr_c)

p_b = clf_b.predict_proba(Xva_c)[:, 1]
acc_b, f1_b, utt_b, call_b = evaluate_calls(ids_va, p_b, yva_c)
save_preds(make_prefix("ecapa", "lrcall", 42), utt_b, call_b)
print(f"(b) 콜 평균    acc={acc_b:.4f}  f1={f1_b:.4f}")

(b) 콜 평균    acc=0.9808  f1=0.9822


In [ ]:
# 로지스틱 회귀 규제 강도 C 비교
for C in [0.01, 0.1, 1, 10]:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    clf.fit(Xtr_c, ytr_c)
    acc, f1, _, _ = evaluate_calls(ids_va, clf.predict_proba(Xva_c)[:, 1], yva_c)
    print(f"C={C:<5} acc={acc:.4f} f1={f1:.4f}")

C=0.01  acc=0.9808 f1=0.9822
C=0.1   acc=0.9813 f1=0.9827
C=1     acc=0.9808 f1=0.9822
C=10    acc=0.9810 f1=0.9824


In [ ]:
# 통화 벡터 집계 방식 비교
def call_stats(s, mode="mean"):
    df = pd.DataFrame(normalize(s["emb"]))
    cols = list(df.columns)
    df["call_id"], df["y"], df["w"] = s["call_id"], s["y"], s["dur"]
    g = df.groupby("call_id")
    if mode == "wmean":   # 발화 길이 가중 평균
        X = g.apply(lambda x: np.average(x[cols], axis=0, weights=x["w"]))
        X = np.stack(X.values)
    elif mode == "meanstd":   # 평균 + 표준편차 (384차원)
        X = np.hstack([g[cols].mean().values, g[cols].std().fillna(0).values])
    else:
        X = g[cols].mean().values
    return X, g["y"].first().values, g.size().index.values

for mode in ["mean", "wmean", "meanstd"]:
    Xtr_m, ytr_m, _ = call_stats(tr, mode)
    Xva_m, yva_m, ids_m = call_stats(va, mode)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    clf.fit(Xtr_m, ytr_m)
    acc, f1, _, _ = evaluate_calls(ids_m, clf.predict_proba(Xva_m)[:, 1], yva_m)
    print(f"{mode:<8} acc={acc:.4f} f1={f1:.4f}")

mean     acc=0.9808 f1=0.9822


wmean    acc=0.9821 f1=0.9834
meanstd  acc=0.9794 f1=0.9809


## 6. 최종 모델 학습 및 저장

길이 가중 평균 + 로지스틱 회귀(C=1)를 Training 전체로 학습합니다.

In [ ]:
def call_wmean(s):
    """발화 임베딩을 발화 길이로 가중 평균해 통화당 벡터 1개 생성 (최종 방식)"""
    df = pd.DataFrame(normalize(s["emb"]))
    cols = list(df.columns)
    df["call_id"], df["y"], df["w"] = s["call_id"], s["y"], s["dur"]
    g = df.groupby("call_id")
    X = np.stack(g.apply(lambda x: np.average(x[cols], axis=0, weights=x["w"]),
                         include_groups=False).values)
    return X, g["y"].first().values, g.size().index.values

Xtr_w, ytr_w, _ = call_wmean(tr)
Xva_w, yva_w, ids_w = call_wmean(va)

clf_w = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000))
clf_w.fit(Xtr_w, ytr_w)
p_w = clf_w.predict_proba(Xva_w)[:, 1]
acc_w, f1_w, utt_w, call_w = evaluate_calls(ids_w, p_w, yva_w)
save_preds(make_prefix("ecapa", "lrwmean", 42), utt_w, call_w)
print(f"저장 완료  acc={acc_w:.4f}  f1={f1_w:.4f}")

저장 완료  acc=0.9821  f1=0.9834


In [ ]:
# 최종 분류기 저장 → 제출 폴더의 weights/clf.joblib
import joblib
joblib.dump(clf_w, os.path.join(DIRS["ckpt"], make_prefix("ecapa", "lrwmean", 42) + "_clf.joblib"))

['/content/drive/MyDrive/DCC/mission1/ckpt/m1_ecapa_lrwmean_s42_clf.joblib']

## 7. 제출용 inference.py 재현 확인

제출 폴더의 `inference.py`를 Validation에 실행해, 학습 때 예측과 같은지 확인합니다.

In [ ]:
SUB_DIR = "/content/drive/MyDrive/DCC/submission/mission1"

!cd "{SUB_DIR}" && python inference.py \
    --audio_dir "{VAL_WAV}" \
    --label_dir "{VAL_JSON}" \
    --ckpt_path weights/clf.joblib \
    --output /content/outputs/mission1.csv

device=cuda, wav 3640개, json 3640개
500/3640 처리
1000/3640 처리
1500/3640 처리
2000/3640 처리
2500/3640 처리
3000/3640 처리
3500/3640 처리
저장: /content/outputs/mission1.csv
전체 추론 시간 664.5초, 통화당 평균 182.5ms


In [ ]:
# 출력 CSV를 정답·학습 시 예측과 비교
out = pd.read_csv("/content/outputs/mission1.csv")
out["call_id"] = out["audio file name"].str.replace(".wav", "", regex=False)

truth = {}
for j in glob.glob(f"{VAL_JSON}/**/*.json", recursive=True):
    with open(j, encoding="utf-8") as f:
        truth[stem(j)] = json.load(f)["gender"]
out["true"] = out["call_id"].map(truth)

print("행 수:", len(out))
print("Accuracy:", round((out["gender"] == out["true"]).mean(), 4))

prev = call_w.set_index("call_id")["pred"].map({1: "F", 0: "M"})
print("학습 때 예측과 일치율:", round((out.set_index("call_id")["gender"] == prev).mean(), 4))
print(out.head(3))

행 수: 3640
Accuracy: 0.9821
학습 때 예측과 일치율: 1.0
                         audio file name gender  \
0  651e464d69a4f266f0626837_20220101.wav      M   
1  651e464d69a4f266f0626845_20220101.wav      M   
2  651e464d69a4f266f0626846_20220101.wav      F   

                             call_id true  
0  651e464d69a4f266f0626837_20220101    M  
1  651e464d69a4f266f0626845_20220101    M  
2  651e464d69a4f266f0626846_20220101    F  


## 8. 계산 효율 정보 (파라미터 수)

Active = 실제 추론에 쓰는 ECAPA 임베딩 모델 + 로지스틱 회귀 (SpeechBrain 패키지의 화자 분류 head는 사용하지 않음)

In [ ]:
import os, joblib
from speechbrain.inference.speaker import EncoderClassifier

SUB_DIR = "/content/drive/MyDrive/DCC/submission/mission1"
ECAPA_DIR = f"{SUB_DIR}/weights/ecapa"

enc = EncoderClassifier.from_hparams(
    source=ECAPA_DIR, savedir="/tmp/ecapa_run",
    overrides={"pretrained_path": ECAPA_DIR}, run_opts={"device": "cpu"})
clf = joblib.load(f"{SUB_DIR}/weights/clf.joblib")

# 모듈별 파라미터 수
for name, m in enc.mods.items():
    print(f"{name:<22}{sum(p.numel() for p in m.parameters()):>12,}")

total = sum(p.numel() for p in enc.mods.parameters())   # 패키지 전체
emb   = sum(p.numel() for p in enc.mods.embedding_model.parameters())
lr    = clf[-1].coef_.size + clf[-1].intercept_.size     # 로지스틱 회귀
print("Total  :", f"{total + lr:,}")
print("Active :", f"{emb + lr:,}", f"({(emb + lr) / (total + lr):.2%})")

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


compute_features                 0
mean_var_norm                    0
embedding_model         20,767,552
mean_var_norm_emb                0
classifier               1,383,360
Total  : 22,151,105
Active : 20,767,745 (93.75%)
